In [2]:
"""
PlacementPredict — Student Clustering: K-Means, Hierarchical, DBSCAN
=====================================================================

Goal:
Cluster students into behavioral/academic "profiles" WITHOUT using the
PlacementStatus label.

PlacementStatus is used only afterwards to describe each cluster
(placement rate per profile), never as a clustering feature.

Outputs:
    - kmeans_elbow_silhouette.png
    - kmeans_pca.png
    - hierarchical_dbscan_pca.png
    - dendrogram.png
    - cluster_profiles.csv
    - placement_predict_with_clusters.csv
"""

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN
)

from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage


# ============================================================
# SETTINGS
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


# ============================================================
# COLUMN DEFINITIONS
# ============================================================

NUMERIC_COLS = [
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
]

CATEGORICAL_COLS = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "ExtraCurricular",
]


# ============================================================
# START
# ============================================================

print("=" * 70)
print("PLACEMENTPREDICT STUDENT CLUSTERING: K-Means / Hierarchical / DBSCAN")
print("=" * 70)


# ============================================================
# 1. LOAD DATA
# ============================================================

# IMPORTANT:
# Keep the CSV file in the same folder as this Python file.

df = pd.read_csv("placement_predict_50k_adjusted (1).csv")


print("\nDataset loaded successfully!")

print("Number of students:", len(df))

print("Number of columns:", len(df.columns))


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = (
    NUMERIC_COLS
    + CATEGORICAL_COLS
    + ["PlacementStatus"]
)

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:

    print("\nERROR: The following columns are missing:")

    for col in missing_columns:
        print("-", col)

    raise ValueError(
        "Please check your CSV column names."
    )


# ============================================================
# 3. PREPROCESS NUMERIC DATA
# ============================================================

print("\nProcessing numeric columns...")

imputer = SimpleImputer(
    strategy="median"
)

num_df = pd.DataFrame(
    imputer.fit_transform(
        df[NUMERIC_COLS]
    ),
    columns=NUMERIC_COLS,
    index=df.index
)


# ============================================================
# 4. PREPROCESS CATEGORICAL DATA
# ============================================================

print("Processing categorical columns...")

cat_df = pd.get_dummies(
    df[CATEGORICAL_COLS],
    drop_first=True
)


# ============================================================
# 5. COMBINE FEATURES
# ============================================================

feature_df = pd.concat(
    [
        num_df,
        cat_df
    ],
    axis=1
)


print(
    "Total clustering features:",
    feature_df.shape[1]
)


# ============================================================
# 6. STANDARDIZE FEATURES
# ============================================================

scaler = StandardScaler()

X = scaler.fit_transform(
    feature_df
)


print(
    f"\nLoaded {len(df):,} students, "
    f"{feature_df.shape[1]} clustering features "
    f"(PlacementStatus excluded)"
)


# ============================================================
# 7. SAFE SAMPLE SIZE
# ============================================================

SAMPLE_SIZE = min(
    5000,
    len(X)
)

sil_sample_idx = (
    np.random.RandomState(
        RANDOM_STATE
    ).choice(
        len(X),
        SAMPLE_SIZE,
        replace=False
    )
)

X_sil_sample = X[sil_sample_idx]


# ============================================================
# 8. CHOOSE K FOR K-MEANS
# ============================================================

print("\nCalculating K-Means models...")

k_range = range(2, 9)

inertias = []

sil_scores = []


for k in k_range:

    km_k = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=RANDOM_STATE
    )

    labels_k = km_k.fit_predict(X)

    inertias.append(
        km_k.inertia_
    )

    sil_scores.append(
        silhouette_score(
            X_sil_sample,
            km_k.predict(X_sil_sample)
        )
    )


# ============================================================
# 9. ELBOW + SILHOUETTE PLOT
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(12, 4.5)
)


ax1.plot(
    list(k_range),
    inertias,
    marker="o"
)

ax1.set_title(
    "Elbow Method (Inertia)"
)

ax1.set_xlabel(
    "Number of Clusters (k)"
)

ax1.set_ylabel(
    "Inertia"
)


ax2.plot(
    list(k_range),
    sil_scores,
    marker="o"
)

ax2.set_title(
    "Silhouette Score vs k"
)

ax2.set_xlabel(
    "Number of Clusters (k)"
)

ax2.set_ylabel(
    "Silhouette Score"
)


plt.tight_layout()

plt.savefig(
    "kmeans_elbow_silhouette.png",
    dpi=150
)

plt.close(fig)


print(
    "Saved plot -> kmeans_elbow_silhouette.png"
)


# ============================================================
# 10. SELECT BEST K
# ============================================================

best_k = list(k_range)[
    int(
        np.argmax(sil_scores)
    )
]


print(
    f"\nSilhouette-suggested k = {best_k}"
)


print(
    "Silhouette scores:"
)


for k, score in zip(
    k_range,
    sil_scores
):

    print(
        f"k={k}: {score:.3f}"
    )


# ============================================================
# 11. K-MEANS ON FULL DATASET
# ============================================================

print(
    f"\nRunning K-Means with k={best_k}..."
)


kmeans = KMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=RANDOM_STATE
)


km_labels = kmeans.fit_predict(
    X
)


# ============================================================
# 12. K-MEANS SILHOUETTE
# ============================================================

sil = silhouette_score(
    X_sil_sample,
    km_labels[sil_sample_idx]
)


print(
    f"K-Means (k={best_k}) silhouette: {sil:.3f}"
)


print(
    "Cluster sizes:",
    pd.Series(
        km_labels
    )
    .value_counts()
    .sort_index()
    .to_dict()
)


# ============================================================
# 13. CLUSTER PROFILES
# ============================================================

profiled = feature_df.copy()


profiled["KMeansCluster"] = km_labels


# PlacementStatus is used ONLY for reporting
profiled["PlacementStatus"] = (
    df["PlacementStatus"].values
)


summary = (
    profiled
    .groupby("KMeansCluster")
    .agg(

        n_students=(
            "PlacementStatus",
            "size"
        ),

        placement_rate=(
            "PlacementStatus",
            "mean"
        ),

        avg_CGPA=(
            "CGPA",
            "mean"
        ),

        avg_AttendancePercent=(
            "AttendancePercent",
            "mean"
        ),

        avg_AptitudeTestScore=(
            "AptitudeTestScore",
            "mean"
        ),

        avg_SoftSkillsRating=(
            "SoftSkillsRating",
            "mean"
        ),

        avg_CodingTestScore=(
            "CodingTestScore",
            "mean"
        ),

        avg_MockInterviewScore=(
            "MockInterviewScore",
            "mean"
        ),

        avg_Internships=(
            "Internships",
            "mean"
        ),

        avg_Projects=(
            "Projects",
            "mean"
        ),
    )
    .round(3)
)


print(
    "\n--- Cluster profiles (KMeansCluster) ---"
)

print(
    summary.to_string()
)


summary.to_csv(
    "cluster_profiles.csv"
)


print(
    "Saved table -> cluster_profiles.csv"
)


# ============================================================
# 14. HIERARCHICAL + DBSCAN
# ============================================================

print(
    "\nRunning Hierarchical Clustering..."
)


sub_idx = (
    np.random.RandomState(
        RANDOM_STATE
    ).choice(
        len(X),
        SAMPLE_SIZE,
        replace=False
    )
)


X_sub = X[sub_idx]


# ============================================================
# 15. HIERARCHICAL CLUSTERING
# ============================================================

hc = AgglomerativeClustering(
    n_clusters=best_k,
    linkage="ward"
)


hc_labels = hc.fit_predict(
    X_sub
)


sil_hc = silhouette_score(
    X_sub,
    hc_labels
)


print(
    f"Hierarchical "
    f"(k={best_k}, subsample n={SAMPLE_SIZE}) "
    f"silhouette: {sil_hc:.3f}"
)


# ============================================================
# 16. DBSCAN
# ============================================================

print(
    "\nRunning DBSCAN..."
)


db = DBSCAN(
    eps=4.0,
    min_samples=15
)


db_labels = db.fit_predict(
    X_sub
)


n_db_clusters = (
    len(
        set(db_labels)
    )
    -
    (
        1
        if -1 in db_labels
        else 0
    )
)


n_noise = int(
    np.sum(
        db_labels == -1
    )
)


print(
    f"DBSCAN (subsample n={SAMPLE_SIZE}): "
    f"{n_db_clusters} clusters, "
    f"{n_noise} noise points"
)


if n_db_clusters >= 2:

    mask = (
        db_labels != -1
    )

    if len(
        set(
            db_labels[mask]
        )
    ) >= 2:

        db_sil = silhouette_score(
            X_sub[mask],
            db_labels[mask]
        )

        print(
            f"DBSCAN silhouette "
            f"(excluding noise): {db_sil:.3f}"
        )


# ============================================================
# 17. DENDROGRAM
# ============================================================

dendro_idx = np.random.choice(
    len(X_sub),
    size=min(
        80,
        len(X_sub)
    ),
    replace=False
)


Z = linkage(
    X_sub[dendro_idx],
    method="ward"
)


plt.figure(
    figsize=(12, 5)
)


dendrogram(
    Z
)


plt.title(
    "Hierarchical Clustering Dendrogram "
    "(Ward Linkage)"
)

plt.xlabel(
    "Student Index"
)

plt.ylabel(
    "Distance"
)


plt.tight_layout()


plt.savefig(
    "dendrogram.png",
    dpi=150
)


plt.close()


print(
    "Saved plot -> dendrogram.png"
)


# ============================================================
# 18. PCA FOR VISUALIZATION
# ============================================================

print(
    "\nPerforming PCA..."
)


X_pca_full = PCA(
    n_components=2
).fit_transform(
    X
)


# ============================================================
# 19. K-MEANS PCA PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(6, 5.2)
)


ax.scatter(
    X_pca_full[:, 0],
    X_pca_full[:, 1],
    c=km_labels,
    cmap="tab10",
    s=8,
    alpha=0.6
)


ax.set_title(
    f"K-Means, Full Data "
    f"({best_k} clusters)"
)

ax.set_xlabel(
    "PC1"
)

ax.set_ylabel(
    "PC2"
)


plt.tight_layout()


plt.savefig(
    "kmeans_pca.png",
    dpi=150
)


plt.close(fig)


print(
    "Saved plot -> kmeans_pca.png"
)


# ============================================================
# 20. HIERARCHICAL + DBSCAN PCA PLOT
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5.2)
)


coords_sub = X_pca_full[
    sub_idx
]


# Hierarchical
axes[0].scatter(
    coords_sub[:, 0],
    coords_sub[:, 1],
    c=hc_labels,
    cmap="tab10",
    s=8,
    alpha=0.6
)


axes[0].set_title(
    f"Hierarchical, "
    f"Subsample ({best_k} clusters)"
)


# DBSCAN
noise_mask = (
    db_labels == -1
)


axes[1].scatter(
    coords_sub[
        ~noise_mask,
        0
    ],
    coords_sub[
        ~noise_mask,
        1
    ],
    c=db_labels[
        ~noise_mask
    ],
    cmap="tab10",
    s=8,
    alpha=0.6
)


if np.any(noise_mask):

    axes[1].scatter(
        coords_sub[
            noise_mask,
            0
        ],
        coords_sub[
            noise_mask,
            1
        ],
        c="lightgray",
        s=8,
        alpha=0.5,
        marker="x",
        label="noise"
    )

    axes[1].legend(
        loc="upper right",
        fontsize=8
    )


axes[1].set_title(
    f"DBSCAN, "
    f"Subsample ({n_db_clusters} clusters)"
)


for ax in axes:

    ax.set_xlabel(
        "PC1"
    )

    ax.set_ylabel(
        "PC2"
    )


plt.tight_layout()


plt.savefig(
    "hierarchical_dbscan_pca.png",
    dpi=150
)


plt.close(fig)


print(
    "Saved plot -> hierarchical_dbscan_pca.png"
)


# ============================================================
# 21. SAVE FINAL DATASET
# ============================================================

df_out = df.copy()


df_out[
    "KMeansCluster"
] = km_labels


df_out.to_csv(
    "placement_predict_with_clusters.csv",
    index=False
)


print(
    "\nSaved deliverable -> "
    "placement_predict_with_clusters.csv"
)


# ============================================================
# 22. FINAL OUTPUT
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "CLUSTERING COMPLETED SUCCESSFULLY"
)

print(
    "=" * 70
)

print(
    "\nGenerated files:"
)

print(
    "1. kmeans_elbow_silhouette.png"
)

print(
    "2. kmeans_pca.png"
)

print(
    "3. hierarchical_dbscan_pca.png"
)

print(
    "4. dendrogram.png"
)

print(
    "5. cluster_profiles.csv"
)

print(
    "6. placement_predict_with_clusters.csv"
)

print(
    "\nDone."
)

PLACEMENTPREDICT STUDENT CLUSTERING: K-Means / Hierarchical / DBSCAN

Dataset loaded successfully!
Number of students: 50000
Number of columns: 21

Processing numeric columns...
Processing categorical columns...
Total clustering features: 33

Loaded 50,000 students, 33 clustering features (PlacementStatus excluded)

Calculating K-Means models...
Saved plot -> kmeans_elbow_silhouette.png

Silhouette-suggested k = 2
Silhouette scores:
k=2: 0.144
k=3: 0.078
k=4: 0.073
k=5: 0.053
k=6: 0.081
k=7: 0.095
k=8: 0.102

Running K-Means with k=2...
K-Means (k=2) silhouette: 0.144
Cluster sizes: {0: 26429, 1: 23571}

--- Cluster profiles (KMeansCluster) ---
               n_students  placement_rate  avg_CGPA  avg_AttendancePercent  avg_AptitudeTestScore  avg_SoftSkillsRating  avg_CodingTestScore  avg_MockInterviewScore  avg_Internships  avg_Projects
KMeansCluster                                                                                                                                          